# 04 Conditional history benchmark

Use the Python kernel. Set analysis_run_dir and raw_root in benchmark_config.json. The annual analysis directory must contain DATA_READY and data/*.parquet.

The workflow preserves focal references, outcomes, team slots, covariates, and fixed effects while reassigning complete author histories. Matching scenarios use broad fields, dominant historical sources, restricted team assignments, and narrower history-length bins.

In [1]:
from pathlib import Path
import sys, importlib
import pandas as pd
PROJECT=Path.cwd().resolve()
assert (PROJECT/"benchmark_config.json").exists(), "Run this notebook from the repository root."
sys.path.insert(0,str(PROJECT))
import benchmark_profiles as profiles
import benchmark_randomization as randomization
importlib.reload(profiles)
importlib.reload(randomization)
MODE="new_clusters"  # new_clusters or extend_existing
assert MODE in {"new_clusters","extend_existing"}
c=profiles.config(PROJECT)
if MODE=="extend_existing":
    previous=Path(c["previous_run_dir"])
    if not previous.is_absolute():previous=PROJECT/previous
    c=profiles.existing_config(previous,PROJECT)
else:
    c=profiles.activate(c)
print("Mode:",MODE,"Run:",c["_out"],"Random worlds:",c["replicates"])

Mode: new_clusters Run: outputs/history_alignment_cluster35_2024 Random worlds: 999


Use new_clusters to prepare the configured member-cluster sample and matching scenarios. Use extend_existing to append random worlds to a compatible completed run. Appending retains the original sample and assignment scheme.

In [2]:
if MODE=="new_clusters":
    prepared=profiles.prepare(c)
    print("Source profiles ready:",prepared)
else:
    print("Reusing existing complete source profiles")

Preparing focal
Focal papers 391239 reference-member clusters 172665
Preparing core
Preparing active_pool
Preparing history_relations
Preparing paper_profiles
Source profiles 1987 (68279, 202752) 320611
Source profiles 1988 (133714, 202752) 642607
Source profiles 1989 (200481, 202752) 961461
Source profiles 1990 (218433, 202752) 1061452
Source profiles 1991 (228289, 202752) 1163138
Source profiles 1992 (242914, 202752) 1269277
Source profiles 1993 (261280, 202752) 1317523
Source profiles 1994 (279717, 202752) 1435876
Source profiles 1995 (301624, 202752) 1564138
Source profiles 1996 (332803, 202752) 1767508
Source profiles 1997 (354308, 202752) 1974060
Source profiles 1998 (376847, 202752) 2172048
Source profiles 1999 (394721, 202752) 2397212
Source profiles 2000 (436635, 202752) 2623334
Source profiles 2001 (467735, 202752) 2845252
Source profiles 2002 (513989, 202752) 3250721
Source profiles 2003 (575798, 202752) 3750702
Source profiles 2004 (647002, 202752) 4240767
Source profiles 2

In [3]:
scenario_paths=randomization.run_scenarios(c)
for path in scenario_paths:
    print(path.name)
    display(pd.read_csv(path/"matching_coverage.csv"))
    if (path/"SCENARIO_EMPTY").exists():print("No complete teams satisfy this scenario; no model will be forced.")

broad 1990 matched 1510 of 1747
broad 1991 matched 1624 of 1872
broad 1992 matched 1822 of 2026
broad 1993 matched 1981 of 2204
broad 1994 matched 2075 of 2301
broad 1995 matched 2178 of 2407
broad 1996 matched 2331 of 2591
broad 1997 matched 2691 of 2939
broad 1998 matched 2893 of 3185
broad 1999 matched 3242 of 3489
broad 2000 matched 3541 of 3817
broad 2001 matched 3740 of 4028
broad 2002 matched 4074 of 4424
broad 2003 matched 4691 of 4978
broad 2004 matched 5094 of 5482
broad 2005 matched 6017 of 6456
broad 2006 matched 6834 of 7235
broad 2007 matched 7313 of 7746
broad 2008 matched 8261 of 8747
broad 2009 matched 9010 of 9515
broad 2010 matched 9844 of 10351
broad 2011 matched 11227 of 11716
broad 2012 matched 12358 of 12800
broad 2013 matched 13542 of 14071
broad 2014 matched 14764 of 15280
broad 2015 matched 16046 of 16681
broad 2016 matched 17339 of 18013
broad 2017 matched 18533 of 19287
broad 2018 matched 19895 of 20581
broad 2019 matched 21782 of 22616
broad 2020 matched 24

,year,initial_papers,matched_papers
0,1990,1747,1510
1,1991,1872,1624
2,1992,2026,1822
3,1993,2204,1981
4,1994,2301,2075
5,1995,2407,2178
6,1996,2591,2331
7,1997,2939,2691
8,1998,3185,2893
9,1999,3489,3242


source_matched


,year,initial_papers,matched_papers
0,1990,1747,239
1,1991,1872,284
2,1992,2026,282
3,1993,2204,343
4,1994,2301,353
5,1995,2407,389
6,1996,2591,441
7,1997,2939,505
8,1998,3185,639
9,1999,3489,668


source_no_team


,year,initial_papers,matched_papers
0,1990,1747,239
1,1991,1872,284
2,1992,2026,281
3,1993,2204,343
4,1994,2301,353
5,1995,2407,378
6,1996,2591,441
7,1997,2939,505
8,1998,3185,639
9,1999,3489,666


narrow_history


,year,initial_papers,matched_papers
0,1990,1747,1359
1,1991,1872,1495
2,1992,2026,1668
3,1993,2204,1805
4,1994,2301,1917
5,1995,2407,2030
6,1996,2591,2140
7,1997,2939,2472
8,1998,3185,2704
9,1999,3489,2994


The default is 999 randomized worlds in addition to the observed world. Computation continues by year and world when matching settings are unchanged. Change run_name for a different sample or input dataset. Default limits are 88 GiB per process and at least 12 GiB of available system memory. Continue with notebook 05 after preparation.